In [ ]:
!pip install -q \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "accelerate>=1.1.1" \
    "sentencepiece" \
    "sacrebleu" \
    "pandas"

!pip install -q "kagglehub[pandas-datasets]"

import os
import kagglehub
import pandas as pd

from kagglehub import KaggleDatasetAdapter

DATASET_HANDLE = (
    "thedevastator/"
    "unlock-the-power-of-english-asl-with-aslg-pc12-c"
)

# Unduh atau gunakan salinan dataset dalam cache KaggleHub
dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)

print("Dataset directory:", dataset_dir)
print("\nFiles available:")

for root, dirs, files in os.walk(dataset_dir):
    for name in files:
        relative_path = os.path.relpath(
            os.path.join(root, name),
            dataset_dir
        )
        print(relative_path)

Using Colab cache for faster access to the 'unlock-the-power-of-english-asl-with-aslg-pc12-c' dataset.
Dataset directory: /kaggle/input/unlock-the-power-of-english-asl-with-aslg-pc12-c

Files available:
train.csv


In [ ]:
# Ganti dengan nama file sebenarnya dari output Cell 1
file_path = "train.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_HANDLE,
    file_path,
)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

/tmp/ipykernel_12600/375044114.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'unlock-the-power-of-english-asl-with-aslg-pc12-c' dataset.
Shape: (87710, 2)
Columns: ['gloss', 'text']


,gloss,text
0,﻿MEMBERSHIP PARLIAMENT SEE MINUTE\n,﻿membership of parliament see minutes\n
1,APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE\n,approval of minutes of previous sitting see mi...
2,MEMBERSHIP PARLIAMENT SEE MINUTE\n,membership of parliament see minutes\n
3,VERIFICATION CREDENTIALS SEE MINUTE\n,verification of credentials see minutes\n
4,DOCUMENT RECEIVE SEE MINUTE\n,documents received see minutes\n


In [ ]:
# Bersihkan teks tanpa mengubah isi kalimat
for col in ["gloss", "text"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Periksa kembali hasilnya
print("Shape:", df.shape)
print("Kolom:", df.columns.tolist())

display(df.head(10))

Shape: (87710, 2)
Kolom: ['gloss', 'text']


,gloss,text
0,MEMBERSHIP PARLIAMENT SEE MINUTE,membership of parliament see minutes
1,APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE,approval of minutes of previous sitting see mi...
2,MEMBERSHIP PARLIAMENT SEE MINUTE,membership of parliament see minutes
3,VERIFICATION CREDENTIALS SEE MINUTE,verification of credentials see minutes
4,DOCUMENT RECEIVE SEE MINUTE,documents received see minutes
5,WRITE STATEMENT AND DESC-ORAL QUESTION TABLE S...,written statements and oral questions tabling ...
6,PETITION SEE MINUTE,petitions see minutes
7,TEXT AGREEMENT DESC-FORWARD BY COUNCIL SEE MINUTE,texts of agreements forwarded by the council s...
8,ACTION TAKE ON PARLIAMENT X-POSS RESOLUTION SE...,action taken on parliament's resolutions see m...
9,AGENDA FOR NEXT SIT SEE MINUTE,agenda for next sitting see minutes


In [ ]:
# Sesuaikan nama kolom dengan kode training kita
if "sentence" not in df.columns and "text" in df.columns:
    df = df.rename(columns={"text": "sentence"})

print("Kolom:", df.columns.tolist())
print("Shape:", df.shape)

display(df[["gloss", "sentence"]].head(10))

Kolom: ['gloss', 'sentence']
Shape: (87710, 2)


,gloss,sentence
0,MEMBERSHIP PARLIAMENT SEE MINUTE,membership of parliament see minutes
1,APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE,approval of minutes of previous sitting see mi...
2,MEMBERSHIP PARLIAMENT SEE MINUTE,membership of parliament see minutes
3,VERIFICATION CREDENTIALS SEE MINUTE,verification of credentials see minutes
4,DOCUMENT RECEIVE SEE MINUTE,documents received see minutes
5,WRITE STATEMENT AND DESC-ORAL QUESTION TABLE S...,written statements and oral questions tabling ...
6,PETITION SEE MINUTE,petitions see minutes
7,TEXT AGREEMENT DESC-FORWARD BY COUNCIL SEE MINUTE,texts of agreements forwarded by the council s...
8,ACTION TAKE ON PARLIAMENT X-POSS RESOLUTION SE...,action taken on parliament's resolutions see m...
9,AGENDA FOR NEXT SIT SEE MINUTE,agenda for next sitting see minutes


In [ ]:
from sklearn.model_selection import train_test_split

# Pastikan kolom yang diperlukan tersedia
assert {"gloss", "sentence"}.issubset(df.columns), (
    f"Kolom tidak sesuai: {df.columns.tolist()}"
)

# Bersihkan karakter BOM dan spasi
for col in ["gloss", "sentence"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

# Hapus data kosong
df = df.dropna(subset=["gloss", "sentence"])
df = df[
    (df["gloss"] != "") &
    (df["sentence"] != "")
]

# Hapus pasangan gloss-kalimat yang identik
before = len(df)
df = df.drop_duplicates(
    subset=["gloss", "sentence"]
).reset_index(drop=True)

print("Duplikat dihapus:", before - len(df))
print("Total pasangan valid:", len(df))

# Bagi menjadi 80% training, 10% validation, 10% testing
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("\nJumlah data:")
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

# Periksa beberapa contoh
display(train_df.head(5))

Duplikat dihapus: 6619
Total pasangan valid: 81091

Jumlah data:
Train      : 64872
Validation : 8109
Test       : 8110


,gloss,sentence
713,"MADAM PRESIDENT , X-I COULD DESC-NOT VOTE IN T...","madam president , I could not vote in time ."
34041,"ON CONTRARY , X-I DESC-ALMOST FEEL SYMPATHY TO...","on the contrary , I almost feel sympathy towar..."
72795,"IN ISLAND X-I HAIL FROM , X-IT BE DESC-MAIN IN...","in the islands I hail from , it is the main in..."
20613,THIS BE BECAUSE X-WE BE TALK DESC-NOT DESC-ONL...,this is because we are talking not only about ...
25997,X-WE BE DESC-REALLY RAISE CAPITALISM X-POSS DE...,we are really raising capitalism's moral stand...


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Model pretrained yang akan kita fine-tune
MODEL_NAME = "google-t5/t5-large"

# Instruksi untuk tugas gloss-to-text
PREFIX = "translate ASL gloss to English: "

# Muat tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ubah DataFrame Pandas menjadi Hugging Face Dataset
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "validation": Dataset.from_pandas(
        val_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
    "test": Dataset.from_pandas(
        test_df[["gloss", "sentence"]].reset_index(drop=True),
        preserve_index=False,
    ),
})

print(raw_datasets)
print("Tokenizer berhasil dimuat:", MODEL_NAME)

DatasetDict({
    train: Dataset({
        features: ['gloss', 'sentence'],
        num_rows: 64872
    })
    validation: Dataset({
        features: ['gloss', 'sentence'],
        num_rows: 8109
    })
    test: Dataset({
        features: ['gloss', 'sentence'],
        num_rows: 8110
    })
})
Tokenizer berhasil dimuat: google-t5/t5-small


In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    # Input model: gloss ASL
    inputs = [
        PREFIX + gloss
        for gloss in examples["gloss"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    # Target model: kalimat bahasa Inggris
    targets = tokenizer(
        text_target=examples["sentence"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = targets["input_ids"]

    return model_inputs


tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["gloss", "sentence"],
    desc="Tokenizing gloss-to-text dataset",
)

print(tokenized_datasets)

Tokenizing gloss-to-text dataset:   0%|          | 0/64872 [00:00<?, ? examples/s]

Tokenizing gloss-to-text dataset:   0%|          | 0/8109 [00:00<?, ? examples/s]

Tokenizing gloss-to-text dataset:   0%|          | 0/8110 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 64872
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8109
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8110
    })
})


In [ ]:
sample = tokenized_datasets["train"][0]

print("Input IDs:")
print(sample["input_ids"])

print("\nDecoded input:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True,
    )
)

print("\nDecoded target:")
print(
    tokenizer.decode(
        sample["labels"],
        skip_special_tokens=True,
    )
)

print("\nJumlah data:")
for split in tokenized_datasets:
    print(split, ":", len(tokenized_datasets[split]))

Input IDs:
[13959, 71, 5629, 20666, 12, 1566, 10, 283, 16759, 329, 3, 22120, 29433, 3, 6, 3, 4, 18, 196, 205, 27986, 309, 25067, 18, 7400, 382, 3, 8040, 3463, 3388, 332, 15382, 3, 5, 1]

Decoded input:
translate ASL gloss to English: MADAM PRESIDENT, X-I COULD DESC-NOT VOTE IN TIME.

Decoded target:
madam president, I could not vote in time.

Jumlah data:
train : 64872
validation : 8109
test : 8110


In [ ]:
import os
import inspect
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# Pastikan runtime Colab menggunakan GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

# Muat model pretrained
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Padding dilakukan secara dinamis dalam setiap batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8,
)

TRAINING_DIR = "/content/asl_t5_training"

args_kwargs = dict(
    output_dir=TRAINING_DIR,
    learning_rate=3e-4,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    warmup_ratio=0.03,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    seed=42,
    dataloader_num_workers=2,
)

# Menyesuaikan nama argumen dengan versi Transformers
training_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

if "eval_strategy" in training_params:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**args_kwargs)

# Menyesuaikan API Trainer dengan versi Transformers
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Model siap dilatih.")
print("Train:", len(tokenized_datasets["train"]))
print("Validation:", len(tokenized_datasets["validation"]))

GPU: Tesla T4


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model siap dilatih.
Train: 64872
Validation: 8109


In [ ]:
train_result = trainer.train()

print("Training selesai.")
print("Training loss:", train_result.training_loss)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.416300,0.291699
2,0.305400,0.233202
3,0.269900,0.224545


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training selesai.
Training loss: 0.4472915160229703


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import shutil

from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

FINAL_MODEL_DIR = "/content/asl_t5_finetuned"

# trainer memuat checkpoint terbaik berdasarkan validation loss
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Salin model ke Google Drive
DRIVE_MODEL_DIR = (
    "/content/drive/MyDrive/ASL/asl_t5_finetuned"
)

os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)

shutil.copytree(
    FINAL_MODEL_DIR,
    DRIVE_MODEL_DIR,
    dirs_exist_ok=True,
)

print("Model lokal :", FINAL_MODEL_DIR)
print("Model Drive :", DRIVE_MODEL_DIR)
print("Isi direktori:", os.listdir(FINAL_MODEL_DIR))

Model lokal : /content/asl_t5_finetuned
Model Drive : /content/drive/MyDrive/ASL/asl_t5_finetuned
Isi direktori: ['tokenizer.json', 'special_tokens_map.json', 'spiece.model', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'config.json', 'generation_config.json']


In [ ]:
import torch
import pandas as pd

from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF

model = trainer.model
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PREFIX = "translate ASL gloss to English: "
MAX_INPUT_LENGTH = 128

test_data = raw_datasets["test"]

predictions = []
references = []
test_glosses = []

BATCH_SIZE = 16

for start in tqdm(
    range(0, len(test_data), BATCH_SIZE),
    desc="Evaluating test set",
):
    end = min(start + BATCH_SIZE, len(test_data))
    batch = test_data[start:end]

    prompts = [
        PREFIX + gloss
        for gloss in batch["gloss"]
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
        )

    batch_predictions = tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )

    predictions.extend(batch_predictions)
    references.extend(batch["sentence"])
    test_glosses.extend(batch["gloss"])

# Hitung metrik
bleu_metric = BLEU(lowercase=True)
chrf_metric = CHRF(word_order=2, lowercase=True)

bleu_score = bleu_metric.corpus_score(
    predictions,
    [references],
)

chrf_score = chrf_metric.corpus_score(
    predictions,
    [references],
)

print(f"Test BLEU: {bleu_score.score:.2f}")
print(f"Test chrF++: {chrf_score.score:.2f}")

# Periksa hasil terjemahan secara manual
results_df = pd.DataFrame({
    "gloss": test_glosses,
    "reference": references,
    "prediction": predictions,
})

display(results_df.head(20))

# Simpan seluruh prediksi agar bisa dianalisis
results_df.to_csv(
    "/content/asl_test_predictions.csv",
    index=False,
)

Evaluating test set:   0%|          | 0/507 [00:00<?, ?it/s]

Test BLEU: 81.01
Test chrF++: 90.83


,gloss,reference,prediction
0,TRANSITION BE DESC-ALWAYS OPPORTUNITY .,transitions are always an opportunity .,the transition is always an opportunity.
1,X-WE NEED TO HAVE MORE AND DESC-BETTER COORDIN...,we need to have more and better coordination w...,we need to have more and better coordination w...
2,X-I DO DESC-NOT SEE EU TAKE LEAD ROLE IN CLIMA...,i do not see the eu taking a leading role in c...,i do not see the eu taking a leading role in c...
3,PROBLEM ABUSE AT HAND THAUS OFFICIAL MUST BE D...,the problems of abuse at the hands of thai off...,the problem of abuse at hand of the thai offic...
4,IN MOST MEMBER STATE DESC-ECONOMIC GROWTH HAVE...,in most member states economic growth has decr...,in most member states economic growth has decr...
5,APPLICATION THIS POLICY CAN DESC-NOT BE IMPROV...,the application of this policy cannot be impro...,the application of this policy cannot be impro...
6,DESC-TRAGIC CIRCUMSTANCE THIS DEATH SHOULD PUT...,the tragic circumstances of this death should ...,the tragic circumstances of this death should ...
7,THAT BE DESC-NOT DESC-UNUSUAL .,that is not unusual .,that is not unusual.
8,X-IT BE DISCUSSION THAT WILL DESC-CERTAINLY CO...,it is a discussion that will certainly contrib...,it is a discussion that will certainly contrib...
9,DEVELOPMENT UNIFY DESC-REGULATORY FRAMEWORK AN...,the development of a unified regulatory framew...,the development of a unified regulatory framew...


In [ ]:
!pip install -q "optimum-onnx[onnxruntime]" onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 23.6 MB/s eta 0:00:00


In [ ]:
%pip uninstall -y diffusers

Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

MODEL_DIR = "/content/drive/MyDrive/ASL/asl_t5_finetuned"

assert os.path.isdir(MODEL_DIR), (
    f"Checkpoint tidak ditemukan: {MODEL_DIR}"
)

print("Checkpoint tersedia:")
print(os.listdir(MODEL_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint tersedia:
['tokenizer.json', 'special_tokens_map.json', 'spiece.model', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'config.json', 'generation_config.json']


In [ ]:
%pip show optimum-onnx
%pip show transformers
%pip show huggingface-hub

Name: optimum-onnx
Version: 0.1.0
Summary: Optimum ONNX is an interface between the Hugging Face libraries and ONNX / ONNX Runtime
Home-page: https://github.com/huggingface/optimum-onnx
Author: 
Author-email: "HuggingFace Inc. Special Ops Team" <hardware@huggingface.co>
License: Apache-2.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: onnx, optimum, transformers
Required-by: 
Name: transformers
Version: 4.46.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: optimum, optimum-onnx, peft, sentence-transform

In [ ]:
import os
import shutil

MODEL_DIR = "/content/drive/MyDrive/ASL/asl_t5_finetuned"
ONNX_DIR = "/content/asl_t5_onnx_opencv"

shutil.rmtree(ONNX_DIR, ignore_errors=True)

!optimum-cli export onnx \
    --model "{MODEL_DIR}" \
    --task text2text-generation \
    --opset 17 \
    "{ONNX_DIR}"

I0000 00:00:1792148174.745786   12903 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792148177.032632   12903 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/optimum/exporters/base.py:151: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self._normalized_config = self.NORMALIZED_CONFIG_CLASS(self._config)
Opset 17 is lower than the recommended minimum opset (18) to export t5. The ONNX export may fail or the exported model may be suboptimal.
/usr/local/lib/python3.13/dist-packages/transformers/models/t5/modeling_t5.py:1306: TracerWarning: Converting a tensor to a Python boolean

In [ ]:
import os

for root, dirs, files in os.walk(ONNX_DIR):
    for filename in files:
        print(os.path.join(root, filename))

/content/asl_t5_onnx_opencv/tokenizer.json
/content/asl_t5_onnx_opencv/special_tokens_map.json
/content/asl_t5_onnx_opencv/decoder_model.onnx
/content/asl_t5_onnx_opencv/spiece.model
/content/asl_t5_onnx_opencv/tokenizer_config.json
/content/asl_t5_onnx_opencv/config.json
/content/asl_t5_onnx_opencv/generation_config.json
/content/asl_t5_onnx_opencv/encoder_model.onnx


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import onnx

ONNX_DIR = Path("/content/asl_t5_onnx_opencv")
DRIVE_ONNX_DIR = Path(
    "/content/drive/MyDrive/ASL/asl_t5_onnx_opencv"
)

assert (ONNX_DIR / "encoder_model.onnx").exists()
assert (ONNX_DIR / "decoder_model.onnx").exists()

# Simpan hasil ekspor ke Google Drive
shutil.copytree(
    ONNX_DIR,
    DRIVE_ONNX_DIR,
    dirs_exist_ok=True,
)

# Validasi kedua graph
for filename in ["encoder_model.onnx", "decoder_model.onnx"]:
    path = DRIVE_ONNX_DIR / filename
    onnx.checker.check_model(str(path))
    print(f"[VALID] {filename} — {path.stat().st_size / 1e6:.2f} MB")

print("\nFolder ONNX tersimpan:", DRIVE_ONNX_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[VALID] encoder_model.onnx — 141.41 MB
[VALID] decoder_model.onnx — 166.69 MB

Folder ONNX tersimpan: /content/drive/MyDrive/ASL/asl_t5_onnx_opencv


In [ ]:
import torch
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM

MODEL_DIR = "/content/drive/MyDrive/ASL/asl_t5_onnx_opencv"

tokenizer_onnx = AutoTokenizer.from_pretrained(MODEL_DIR)

model_onnx = ORTModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    provider="CPUExecutionProvider",
    use_cache=False,
)

gloss = "MEMBERSHIP PARLIAMENT SEE MINUTE"
prompt = "translate ASL gloss to English: " + gloss

inputs = tokenizer_onnx(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

with torch.inference_mode():
    output_ids = model_onnx.generate(
        **inputs,
        max_new_tokens=48,
        num_beams=2,
        do_sample=False,
        use_cache=False,
    )

prediction = tokenizer_onnx.decode(
    output_ids[0],
    skip_special_tokens=True,
)

print("Input gloss :", gloss)
print("Prediksi    :", prediction)

Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('decoder_model.onnx'), PosixPath('encoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


Input gloss : MEMBERSHIP PARLIAMENT SEE MINUTE
Prediksi    : membership of parliament see minutes


In [ ]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

MODEL_DIR = "/content/drive/MyDrive/ASL/asl_t5_onnx_opencv"

tokenizer_onnx = AutoTokenizer.from_pretrained(MODEL_DIR)

model_onnx = ORTModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    provider="CPUExecutionProvider",
    use_cache=False,
    use_merged=False,
)

print("ONNX model loaded successfully.")

ONNX model loaded successfully.


In [ ]:
import torch

PREFIX = "translate ASL gloss to English: "

for i in range(min(5, len(raw_datasets["test"]))):
    sample = raw_datasets["test"][i]

    inputs = tokenizer_onnx(
        PREFIX + sample["gloss"],
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    with torch.inference_mode():
        output_ids = model_onnx.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=2,
            do_sample=False,
            use_cache=False,
        )

    prediction = tokenizer_onnx.decode(
        output_ids[0],
        skip_special_tokens=True,
    )

    print(f"\nContoh {i + 1}")
    print("Gloss :", sample["gloss"])
    print("Target:", sample["sentence"])
    print("Prediksi:", prediction)


Contoh 1
Gloss : TRANSITION BE DESC-ALWAYS OPPORTUNITY .
Target: transitions are always an opportunity .
Prediksi: the transition is always an opportunity.

Contoh 2
Gloss : X-WE NEED TO HAVE MORE AND DESC-BETTER COORDINATION WITH EUROPOL .
Target: we need to have more and better coordination with europol .
Prediksi: we need to have more and better coordination with europol.

Contoh 3
Gloss : X-I DO DESC-NOT SEE EU TAKE LEAD ROLE IN CLIMATE PROTECTION AND IN DEVELOPMENT AID .
Target: i do not see the eu taking a leading role in climate protection and in development aid .
Prediksi: i do not see the eu taking a leading role in climate protection and in development aid.

Contoh 4
Gloss : PROBLEM ABUSE AT HAND THAUS OFFICIAL MUST BE DESC-SERIOUSLY ADDRESS .
Target: the problems of abuse at the hands of thai officials must be seriously addressed .
Prediksi: the problem of abuse at hand of the thai officials must be seriously addressed.

Contoh 5
Gloss : IN MOST MEMBER STATE DESC-ECONOMIC G

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# Folder hasil ekspor ONNX
MODEL_DIR = Path("/content/asl_t5_onnx_opencv")

# Pastikan file utama tersedia
required_files = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "spiece.model",
]

missing = [
    name for name in required_files
    if not (MODEL_DIR / name).is_file()
]

if missing:
    raise FileNotFoundError(
        "File berikut tidak ditemukan:\n"
        + "\n".join(missing)
    )

# Buat satu ZIP yang berisi seluruh folder model
ZIP_PATH = Path("/content/asl_t5_onnx_opencv.zip")

with ZipFile(ZIP_PATH, "w", ZIP_DEFLATED) as zf:
    for path in MODEL_DIR.rglob("*"):
        if path.is_file():
            # Pertahankan struktur folder di dalam ZIP
            arcname = Path(MODEL_DIR.name) / path.relative_to(MODEL_DIR)
            zf.write(path, arcname)

print("ZIP berhasil dibuat!")
print(f"Jumlah file: {sum(1 for p in MODEL_DIR.rglob('*') if p.is_file())}")
print(f"Ukuran ZIP: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

# Mulai unduhan ke komputer melalui browser
files.download(str(ZIP_PATH))

ZIP berhasil dibuat!
Jumlah file: 8
Ukuran ZIP: 273.30 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import torch

onnx_tokenizer = AutoTokenizer.from_pretrained(ONNX_DIR)

onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    ONNX_DIR,
    provider="CPUExecutionProvider",
)

sample = raw_datasets["test"][0]
gloss = sample["gloss"]

inputs = onnx_tokenizer(
    PREFIX + gloss,
    return_tensors="pt",
    truncation=True,
    max_length=128,
)

with torch.inference_mode():
    output_ids = onnx_model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=2,
        do_sample=False,
    )

prediction = onnx_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
)

print("Input gloss :", gloss)
print("Target      :", sample["sentence"])
print("ONNX output :", prediction)

In [ ]:
DRIVE_ONNX_DIR = (
    "/content/drive/MyDrive/ASL/asl_t5_onnx_opencv"
)

shutil.copytree(
    ONNX_DIR,
    DRIVE_ONNX_DIR,
    dirs_exist_ok=True,
)

print("ONNX tersimpan di:", DRIVE_ONNX_DIR)